# Lab · Huấn luyện theo kiểu PyTorch

Bài tập cho [`lesson.md`](lesson.md) (Module 4). Gom bốn lab của khóa theo thứ tự: **A** SGD và `DataLoader`, **B** mini-batch, **C** `optim.SGD`, **D** tách train/validation và chọn learning rate.

**Cách làm:** tự thử trước. Kẹt thì mở **Gợi ý 1**, rồi **Gợi ý 2**, cuối cùng mới **Gợi ý 3**. Lời giải ở thư mục `solution/`.

Môi trường do **uv** quản lý: trong thư mục khóa `04-intro-neural-networks-pytorch/`, chạy `uv sync` (một lần), rồi `uv run jupyter lab`.

## Chuẩn bị

In [ ]:
import os, tempfile
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
%matplotlib inline

print("PyTorch", torch.__version__)

## Bài 1. A · Dataset và `DataLoader` với batch size 1

Viết lớp `Data(Dataset)`: `self.x = torch.arange(-3, 3, 0.1).view(-1, 1)`, `self.f = 1 * self.x - 1`, `self.y = self.f + 0.1 * torch.randn(self.x.size())`, `self.len = self.x.shape[0]`; `__getitem__` trả `(x[i], y[i])`, `__len__` trả `self.len`.

Đặt `torch.manual_seed(1)` rồi tạo `dataset = Data()`. Tạo `loader1 = DataLoader(dataset, batch_size=1)`.
Lưu `iters1 = len(loader1)` và `first_batch = next(iter(loader1))`.

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

`DataLoader` đứng trên một `Dataset` đã đủ `__len__` và `__getitem__`. Số lô bạn nhận được phụ thuộc số mẫu và kích thước lô.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

`torch.manual_seed(1)` trước khi tạo `Data()` để dữ liệu giống nhau mỗi lần. `len(loader)` là số lô. `next(iter(loader))` lấy lô đầu.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

`loader1 = DataLoader(dataset=dataset, batch_size=___)`; `first_batch = next(iter(loader1))`.

</details>

In [ ]:
class Data(Dataset):
    pass  # TODO

torch.manual_seed(1)
dataset = None  # TODO
loader1 = None  # TODO
iters1 = None   # TODO
first_batch = None  # TODO
print(iters1, first_batch)

In [ ]:
assert len(dataset) == 60
assert iters1 == 60, "batch size 1 → 60 iteration mỗi epoch"
x0, y0 = first_batch
assert tuple(x0.shape) == (1, 1) and tuple(y0.shape) == (1, 1), "mỗi lô có thêm chiều batch"
assert abs(x0.item() + 3.0) < 1e-5
print("✅ Bài 1 đúng")

## Bài 2. A · Huấn luyện SGD bằng tay với `DataLoader`

Viết `criterion(yhat, y)` = MSE, và hàm `train_manual(batch_size, epochs=10, lr=0.1)`:
- tạo `loader = DataLoader(dataset, batch_size=batch_size)`;
- `w = -15.0`, `b = -10.0` (có gradient);
- mỗi **epoch**: trước hết ghi cost trên **toàn bộ** dữ liệu `criterion(w * dataset.x + b, dataset.y)` vào list `LOSS`; rồi duyệt từng lô: dự đoán, loss, `backward()`, cập nhật dưới `torch.no_grad()`, đặt gradient về 0;
- trả về `(w.item(), b.item(), LOSS)`.

Chạy với `batch_size=1` (đó là SGD): lưu `w_sgd, b_sgd, LOSS_sgd`.

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Khung giống bài trước, nhưng vòng trong bây giờ duyệt các lô thay vì cả tập một lần.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

Cost cả tập ghi **ngoài** vòng lô, đầu mỗi epoch. Cập nhật: `w -= lr * w.grad`, `b -= lr * b.grad` trong `torch.no_grad()`.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

`for x, y in loader: loss = criterion(w * x + b, y); loss.backward(); with torch.no_grad(): ...`.

</details>

In [ ]:
def criterion(yhat, y):
    pass  # TODO

def train_manual(batch_size, epochs=10, lr=0.1):
    pass  # TODO

w_sgd, b_sgd, LOSS_sgd = None, None, None  # TODO: batch_size = 1
print(w_sgd, b_sgd, LOSS_sgd)

In [ ]:
assert len(LOSS_sgd) == 10
assert LOSS_sgd[0] > 800 and LOSS_sgd[-1] < 0.1
assert abs(w_sgd - 1.0) < 0.2 and abs(b_sgd + 1.0) < 0.2
print("✅ Bài 2 đúng: SGD hội tụ chỉ sau vài epoch")

## Bài 3. A · So với batch gradient descent

Huấn luyện với `batch_size = len(dataset)` (mỗi lần dùng **cả tập**: đó là batch GD), cùng 10 epoch, `lr = 0.1`. Lưu `LOSS_bgd`.
Vẽ `LOSS_bgd` và `LOSS_sgd`. Lưu `bgd_slower = LOSS_bgd[-1] > LOSS_sgd[-1]`.
Câu hỏi: vì sao SGD tiến nhanh hơn tính theo epoch? (Gợi ý: đếm số lần cập nhật.)

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Cùng 10 epoch, nhưng mỗi phương pháp cập nhật tham số bao nhiêu lần?

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

`train_manual(len(dataset))`. Batch GD: 10 lần cập nhật; SGD: 600 lần.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

`_, _, LOSS_bgd = train_manual(batch_size=___)`.

</details>

In [ ]:
LOSS_bgd = None     # TODO
bgd_slower = None   # TODO
print(bgd_slower)

In [ ]:
assert bgd_slower is True
print("✅ Bài 3 đúng: SGD cập nhật 600 lần, batch GD chỉ 10 lần trong cùng 10 epoch")

## Bài 4. B · Đếm iteration và lô cuối

Lập dictionary `iters` cho các batch size `1, 5, 10, 20`: giá trị là số lô (iteration mỗi epoch) của `DataLoader(dataset, batch_size=bs)`.
Rồi với `batch_size=7`:
- `n7`: số lô;
- `n7_drop`: số lô khi dùng `drop_last=True`;
- `last_len`: số mẫu trong lô **cuối** (khi không `drop_last`).

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Khi số mẫu không chia hết cho batch size, phần dư tạo thêm một lô nhỏ ở cuối.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

`len(DataLoader(...))` cho số lô. Để biết lô cuối, duyệt qua loader và nhớ độ dài lô gần nhất: `len(x)`.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

`for x, y in DataLoader(dataset, batch_size=7): last_len = len(x)`.

</details>

In [ ]:
iters = {}      # TODO
n7 = None       # TODO
n7_drop = None  # TODO
last_len = None # TODO
print(iters, n7, n7_drop, last_len)

In [ ]:
assert iters == {1: 60, 5: 12, 10: 6, 20: 3}
assert n7 == 9 and n7_drop == 8, "60 = 8×7 + 4 nên có 9 lô, lô cuối 4 mẫu"
assert last_len == 4
print("✅ Bài 4 đúng")

## Bài 5. B · Mini-batch với batch size 5, 10, 20

Huấn luyện mini-batch với `batch_size` = 5, 10, 20 (10 epoch, `lr = 0.1`). Lưu `final_loss = {5: ..., 10: ..., 20: ...}` (cost cuối của từng cách) và `updates = {bs: số lần cập nhật}` (iteration mỗi epoch × 10).
Vẽ cả năm đường loss: batch GD, SGD, mini-batch 5, 10, 20.

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Bạn đã có sẵn hàm huấn luyện chung. Mỗi lần chỉ đổi một tham số.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

Gọi `train_manual(bs)` cho từng `bs`, lấy phần tử cuối của list loss. Số lần cập nhật = số lô × số epoch.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

`_, _, L = train_manual(bs)`; `final_loss[bs] = L[-1]`.

</details>

In [ ]:
final_loss = {}  # TODO
updates = {}     # TODO
print(final_loss, updates)

In [ ]:
assert set(final_loss) == {5, 10, 20}
assert all(v < 0.05 for v in final_loss.values()), "cả ba đều hội tụ tốt trong 10 epoch"
assert updates == {5: 120, 10: 60, 20: 30}
print("✅ Bài 5 đúng")

## Bài 6. C · Mô hình, hàm loss và optimizer

Viết lớp `linear_regression(nn.Module)` (như Module 3), rồi tạo:
- `model = linear_regression(1, 1)`;
- `loss_fn = nn.MSELoss()`;
- `optimizer = optim.SGD(model.parameters(), lr=0.01)`.

Đặt tham số khởi đầu `weight = -15`, `bias = -10` bằng `model.state_dict()['linear.weight'][0] = ...` (và tương tự cho bias).
Lưu `lr_in_opt` là learning rate đọc từ `optimizer.state_dict()['param_groups'][0]['lr']`.

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Optimizer cần biết hai điều: tham số nào cần cập nhật, và cập nhật với bước bao nhiêu.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

`optim.SGD(model.parameters(), lr=0.01)`. `state_dict()['param_groups']` là list các nhóm tham số, mỗi nhóm có khóa `'lr'`.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

`model.state_dict()['linear.weight'][0] = -15`; `lr_in_opt = optimizer.state_dict()['param_groups'][0]['____']`.

</details>

In [ ]:
class linear_regression(nn.Module):
    pass  # TODO

model = None      # TODO
loss_fn = None    # TODO
optimizer = None  # TODO
# TODO: đặt weight = -15, bias = -10
lr_in_opt = None  # TODO
print(list(model.parameters()), lr_in_opt)

In [ ]:
assert lr_in_opt == 0.01
assert model.linear.weight.item() == -15 and model.linear.bias.item() == -10
print("✅ Bài 6 đúng")

## Bài 7. C · Vòng lặp huấn luyện với optimizer

Huấn luyện `model` ở bài 6 trên `DataLoader(dataset, batch_size=1)`, 10 epoch, dùng đúng năm bước: forward, loss, `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()`.
Sau đó lưu `w_opt = model.linear.weight.item()` và `b_opt = model.linear.bias.item()`.

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Bạn không còn phải tự trừ gradient. Ba dòng cuối của vòng lặp làm hết việc đó.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

Thứ tự: `yhat = model(x)`, `loss = loss_fn(yhat, y)`, `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()`.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

Vòng ngoài `for epoch in range(10)`, vòng trong `for x, y in DataLoader(dataset, batch_size=1)`.

</details>

In [ ]:
# TODO: vòng lặp huấn luyện
w_opt = None  # TODO
b_opt = None  # TODO
print(w_opt, b_opt)

In [ ]:
assert abs(w_opt - 1.0) < 0.15 and abs(b_opt + 1.0) < 0.15, "tham số phải gần w = 1, b = -1"
print("✅ Bài 7 đúng: cùng kết quả với vòng lặp viết tay, nhưng gọn hơn")

## Bài 8. C · Luyện tập: batch GD bằng optimizer

Tạo mô hình **mới** `model_b` (đặt weight −15, bias −10 như trên) và `optimizer_b = optim.SGD(model_b.parameters(), lr=0.1)`.
Huấn luyện **batch GD** (lô là cả tập) 10 epoch. Lưu `cost_b`: MSE của `model_b` trên toàn bộ `dataset.x`, `dataset.y` (không cần gradient).

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Batch GD là trường hợp `DataLoader` chia ra đúng một lô.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

`DataLoader(dataset, batch_size=len(dataset))`. Tính cost cuối dưới `torch.no_grad()`.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

`cost_b = loss_fn(model_b(dataset.x), dataset.y).item()` đặt trong `with torch.no_grad():`.

</details>

In [ ]:
model_b = None      # TODO
optimizer_b = None  # TODO
# TODO: huấn luyện
cost_b = None       # TODO
print(cost_b)

In [ ]:
assert cost_b < 2.0, "sau 10 lần cập nhật cost đã giảm từ ~835 xuống còn vài đơn vị"
assert cost_b > 0.05, "batch GD 10 epoch vẫn chưa hội tụ hẳn như SGD"
print("✅ Bài 8 đúng")

## Bài 9. D · Dữ liệu train có ngoại lai, validation sạch

Viết lớp `DataSplit(Dataset)` với tham số `train=True`:
`x = torch.arange(-3, 3, 0.1).view(-1, 1)`, `f = -3 * x + 1`, `y = f + 0.1 * torch.randn(x.size())`.
Nếu `train`: tạo **ngoại lai** bằng `y[0] = 0` và `y[50:55] = 20`; nếu không thì để nguyên.
Đặt `torch.manual_seed(1)`, tạo `train_data = DataSplit()` và `val_data = DataSplit(train=False)`.

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Hai tập sinh từ cùng một quy luật, nhưng tập huấn luyện bị làm bẩn một cách có chủ ý.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

Gán slice cho tensor: `self.y[50:55] = 20`. Tham số `train` quyết định có làm bẩn hay không.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

`if train: self.y[0] = 0; self.y[50:55] = ___`.

</details>

In [ ]:
class DataSplit(Dataset):
    pass  # TODO

torch.manual_seed(1)
train_data = None  # TODO
val_data = None    # TODO

In [ ]:
assert (train_data.y[50:55] == 20).all() and train_data.y[0].item() == 0
assert (val_data.y[50:55] < 0).all(), "validation không có ngoại lai"
print("✅ Bài 9 đúng")

## Bài 10. D · Chọn learning rate bằng validation

Với `learning_rates = [0.0001, 0.001, 0.01, 0.1]`: với mỗi giá trị, **đặt `torch.manual_seed(1)`**, tạo mô hình `linear_regression(1, 1)`, `optim.SGD(..., lr=lr)`, huấn luyện 10 epoch trên `DataLoader(train_data, batch_size=1)` (loss MSE, năm bước chuẩn).
Sau đó ghi **training error** (MSE trên toàn `train_data`) và **validation error** (MSE trên toàn `val_data`, dưới `torch.no_grad()`) vào hai tensor `train_error`, `validation_error` (độ dài 4), và lưu mô hình vào list `MODELS`.
Rồi:
- `best_val_idx`: chỉ số có validation error nhỏ nhất;
- `best_train_idx`: chỉ số có training error nhỏ nhất.
**Đoán trước:** hai chỉ số này có trùng nhau không?

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Đặt seed trước mỗi mô hình để chênh lệch kết quả là do learning rate chứ không do khởi tạo ngẫu nhiên.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

Hai vòng `for`: ngoài là learning rate; trong là epoch × lô. Chọn chỉ số nhỏ nhất: `tensor.argmin()`.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

`validation_error[i] = loss_fn(model(val_data.x), val_data.y).item()` trong `torch.no_grad()`.

</details>

In [ ]:
learning_rates = [0.0001, 0.001, 0.01, 0.1]
train_error = torch.zeros(len(learning_rates))
validation_error = torch.zeros(len(learning_rates))
MODELS = []
# TODO: huấn luyện và ghi lại sai số
best_val_idx = None    # TODO
best_train_idx = None  # TODO
print(train_error, validation_error)

In [ ]:
assert len(MODELS) == 4
assert best_val_idx == int(validation_error.argmin())
assert validation_error[0] > validation_error[best_val_idx], "lr = 0.0001 quá nhỏ: học chưa kịp"
print("best theo train:", learning_rates[best_train_idx], "| best theo validation:", learning_rates[best_val_idx])
print("✅ Bài 10 đúng. Chọn mô hình theo VALIDATION, không theo training.")

## Bài 11. D · Lưu và nạp mô hình tốt nhất

Lấy `best_model = MODELS[best_val_idx]`.
- Lưu `state_dict` vào file tạm: `path = os.path.join(tempfile.gettempdir(), "best_model.pt")`, bằng `torch.save`.
- Tạo `loaded = linear_regression(1, 1)` (tham số khác), nạp tham số từ file bằng `load_state_dict`, rồi `.eval()`.
- `same = True` nếu `loaded` và `best_model` cho **cùng** dự đoán trên `val_data.x`.

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Mô hình mới cần cùng kiến trúc thì mới nhận được tham số đã lưu. Dữ liệu được lưu là dictionary tham số, không phải cả lớp.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

`torch.save(obj, path)`, `torch.load(path)`, `model.load_state_dict(dict)`, `model.eval()`. So sánh dự đoán bằng `torch.allclose`.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

`loaded.load_state_dict(torch.load(path))`.

</details>

In [ ]:
best_model = None  # TODO
path = os.path.join(tempfile.gettempdir(), "best_model.pt")
# TODO: lưu, tạo loaded, nạp, eval
same = None  # TODO
print(same)

In [ ]:
assert os.path.exists(path)
assert same is True
print("✅ Bài 11 đúng: không cần huấn luyện lại để dùng mô hình")

## Bài 12. D · Đánh giá cuối: không cập nhật, không theo dõi gradient

Tính `test_loss`: MSE của `best_model` trên `val_data` (coi đây là bước đánh giá) **bên trong** `torch.no_grad()`.
Rồi lưu `no_graph = not test_loss.requires_grad`.
Câu hỏi: nếu không bọc `no_grad`, kết quả có gì khác? Vì sao khi đánh giá ta không gọi `backward()` và `step()`?

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Đánh giá chỉ để đo, không để học. Những thứ phục vụ việc học (đồ thị gradient) là lãng phí.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

`with torch.no_grad(): ...`. Tensor kết quả tính trong khối này không có `requires_grad`.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

`test_loss = loss_fn(best_model(val_data.x), val_data.y)` bên trong khối `with`.

</details>

In [ ]:
test_loss = None  # TODO
no_graph = None   # TODO
print(test_loss, no_graph)

In [ ]:
assert no_graph is True
assert test_loss.item() == validation_error[best_val_idx].item()
print("✅ Bài 12 đúng")

## Xong rồi
Đối chiếu với `solution/`, rồi quay lại `lesson.md` nếu còn chỗ chưa chắc.